In [1]:
import sys
print(sys.executable)
import gpaw, ase
print("gpaw", gpaw.__version__, "| ase", ase.__version__)

/home/buyer/miniconda3/envs/dft/bin/python
gpaw 25.7.0 | ase 3.29.0


In [2]:
from ase import Atoms
from gpaw import GPAW, PW
import time

a = 3.905  # experimental cubic lattice constant, Angstroms

srtio3 = Atoms(
    symbols='SrTiO3',
    scaled_positions=[
        (0.0, 0.0, 0.0),   # Sr - corner
        (0.5, 0.5, 0.5),   # Ti - body center
        (0.5, 0.5, 0.0),   # O - face center
        (0.5, 0.0, 0.5),   # O - face center
        (0.0, 0.5, 0.5),   # O - face center
    ],
    cell=[a, a, a],
    pbc=True,
)
print(srtio3)

srtio3.calc = GPAW(xc='PBE', mode=PW(400), kpts=(6, 6, 6), txt=None)
t0 = time.time()
e = srtio3.get_potential_energy()
print(f"total energy: {e:.4f} eV   ({time.time()-t0:.0f} s)")

Atoms(symbols='SrTiO3', pbc=True, cell=[3.905, 3.905, 3.905])
total energy: -33.9594 eV   (7 s)


In [3]:
for ec in [300, 400, 500, 600, 700]:
    srtio3.calc = GPAW(xc='PBE', mode=PW(ec), kpts=(6, 6, 6), txt=None)
    e = srtio3.get_potential_energy()
    print(f"cutoff {ec} eV -> {e:.4f} eV")

cutoff 300 eV -> -18.2151 eV
cutoff 400 eV -> -33.9594 eV
cutoff 500 eV -> -38.2873 eV
cutoff 600 eV -> -39.3420 eV
cutoff 700 eV -> -39.5761 eV


In [4]:
for ec in [800, 900, 1000, 1100]:
    srtio3.calc = GPAW(xc='PBE', mode=PW(ec), kpts=(6, 6, 6), txt=None)
    e = srtio3.get_potential_energy()
    print(f"cutoff {ec} eV -> {e:.4f} eV")

cutoff 800 eV -> -39.6224 eV
cutoff 900 eV -> -39.6300 eV
cutoff 1000 eV -> -39.6318 eV
cutoff 1100 eV -> -39.6331 eV


In [5]:
for k in [3, 4, 5, 6, 7, 8]:
    srtio3.calc = GPAW(xc='PBE', mode=PW(900), kpts=(k, k, k), txt=None)
    e = srtio3.get_potential_energy()
    print(f"k-grid {k}x{k}x{k} -> {e:.4f} eV")

k-grid 3x3x3 -> -39.5257 eV
k-grid 4x4x4 -> -39.6440 eV
k-grid 5x5x5 -> -39.6260 eV
k-grid 6x6x6 -> -39.6300 eV
k-grid 7x7x7 -> -39.6292 eV
k-grid 8x8x8 -> -39.6294 eV


In [6]:
from ase.dft.bandgap import bandgap

srtio3_final = Atoms(
    symbols='SrTiO3',
    scaled_positions=[(0,0,0), (0.5,0.5,0.5), (0.5,0.5,0), (0.5,0,0.5), (0,0.5,0.5)],
    cell=[3.905, 3.905, 3.905],
    pbc=True,
)
srtio3_final.calc = GPAW(xc='PBE', mode=PW(900), kpts=(8, 8, 8), txt=None)
srtio3_final.get_potential_energy()

gap, p1, p2 = bandgap(srtio3_final.calc)
print(f"PBE gap: {gap:.3f} eV")
print(f"experimental gap: 3.20 eV")
print(f"underestimation: {100*(3.20 - gap)/3.20:.0f}%")

PBE gap: 2.110 eV
experimental gap: 3.20 eV
underestimation: 34%


## Phase 1 findings — SrTiO3, the clean demonstration
- Converged settings: PW 900 eV (highest of any material so far — oxygen's compact 2p orbitals need fine plane-wave detail, a DIFFERENT reason for a high cutoff than lead's heaviness in CsPbI3), k-grid 6×6×6.
- **PBE gap: 2.110 eV vs. experimental 3.20 eV → -34% underestimation.** Textbook PBE failure, correct direction, expected magnitude — the clean demo Phase 1 originally wanted.
- Reference values (lattice 3.905 Å, gap ~3.2-3.25 eV) verified against 5 independent literature sources before starting.
- Combined with the CsPbI3 result: PBE's gap error is NOT one fixed number. It's a systematic underestimation that can be REVERSED by missing spin-orbit coupling in heavy-element materials. A single "correction factor" isn't trustworthy without knowing whether SOC matters for the specific material.